# Building Language Models: From N-grams to Transformers

This notebook takes you on a hands-on journey through the evolution of language models. We'll **build working examples** of each major approach to understand why transformers became dominant.

## Learning Objectives

- Build and compare different language model architectures
- Understand the limitations that drove each innovation
- See how attention solves problems that RNNs couldn't
- Build a mini "GPT-style" transformer from scratch

## What We'll Build

| Era | Model | What We'll Code |
|-----|-------|-----------------|
| Statistical NLP | N-gram LM | Bigram/trigram text generator |
| Neural Embeddings | Word2Vec | Skip-gram word vectors |
| Sequence Models | RNN/LSTM | Character-level RNN |
| Transformers | Attention | Mini GPT with self-attention |

# Setup: Install dependencies if needed
# Uncomment and run if you need to install packages

# !pip install numpy matplotlib torch

In [ ]:
import numpy as np
import random
from collections import Counter, defaultdict
import math

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)

---

## Part 1: N-gram Language Models (Statistical NLP Era)

**The idea**: Predict the next word based on the previous N-1 words.

**Key insight**: Language has patterns. "I want to" is more likely followed by "eat" than "elephant".

**Limitation**: No understanding of meaning. "The cat sat on the mat" and "The mat sat on the cat" have the same word frequencies but different meanings.

In [ ]:
class NgramLM:
    """
    N-gram Language Model.
    
    Predicts P(next_word | previous N-1 words) by counting occurrences in training data.
    Uses Laplace smoothing to handle unseen n-grams.
    """
    
    def __init__(self, n: int = 2, smoothing: float = 1.0):
        self.n = n  # n=2 is bigram, n=3 is trigram
        self.smoothing = smoothing
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()
    
    def _get_ngrams(self, tokens: list[str]) -> list[tuple]:
        """Extract n-grams from a token list."""
        # Add start/end tokens
        padded = ['<s>'] * (self.n - 1) + tokens + ['</s>']
        ngrams = []
        for i in range(len(padded) - self.n + 1):
            ngram = tuple(padded[i:i + self.n])
            ngrams.append(ngram)
        return ngrams
    
    def fit(self, texts: list[str]) -> None:
        """Train on a list of texts."""
        for text in texts:
            tokens = text.lower().split()
            self.vocab.update(tokens)
            self.vocab.add('<s>')
            self.vocab.add('</s>')
            
            for ngram in self._get_ngrams(tokens):
                context = ngram[:-1]  # All but last word
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
    
    def prob(self, word: str, context: tuple) -> float:
        """P(word | context) with Laplace smoothing."""
        ngram = context + (word,)
        V = len(self.vocab)
        return (self.ngram_counts[ngram] + self.smoothing) / \
               (self.context_counts[context] + self.smoothing * V)
    
    def generate(self, max_words: int = 20, start: str = None) -> str:
        """Generate text by sampling from the model."""
        if start:
            tokens = start.lower().split()
        else:
            tokens = []
        
        context = ['<s>'] * (self.n - 1)
        if tokens:
            context = context[:-len(tokens)] + tokens if len(tokens) < self.n - 1 else tokens[-(self.n-1):]
        
        for _ in range(max_words):
            # Get probabilities for all words
            probs = {w: self.prob(w, tuple(context)) for w in self.vocab}
            
            # Sample from distribution
            words = list(probs.keys())
            weights = list(probs.values())
            next_word = random.choices(words, weights=weights)[0]
            
            if next_word == '</s>':
                break
            
            tokens.append(next_word)
            context = context[1:] + [next_word]
        
        return ' '.join(tokens)
    
    def perplexity(self, texts: list[str]) -> float:
        """Calculate perplexity on test texts. Lower is better."""
        total_log_prob = 0
        total_words = 0
        
        for text in texts:
            tokens = text.lower().split()
            for ngram in self._get_ngrams(tokens):
                context = ngram[:-1]
                word = ngram[-1]
                prob = self.prob(word, context)
                total_log_prob += math.log(prob)
                total_words += 1
        
        return math.exp(-total_log_prob / total_words)

# Training corpus - simple sentences about AI/ML
training_corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "I like to learn machine learning",
    "deep learning is a subset of machine learning",
    "neural networks learn from data",
    "the model learns patterns from examples",
    "attention is all you need",
    "transformers use self attention mechanisms",
    "language models predict the next word",
    "large language models are trained on text",
]

# Train bigram and trigram models
bigram_lm = NgramLM(n=2)
bigram_lm.fit(training_corpus)

trigram_lm = NgramLM(n=3)
trigram_lm.fit(training_corpus)

print("=== Bigram Model (n=2) ===")
print("Looks at 1 previous word\n")
for i in range(5):
    print(f"  {i+1}. {bigram_lm.generate(max_words=10)}")

print("\n=== Trigram Model (n=3) ===")
print("Looks at 2 previous words\n")
for i in range(5):
    print(f"  {i+1}. {trigram_lm.generate(max_words=10)}")

### N-gram Limitations

Notice how the generated text:
- Sometimes makes grammatical sense locally but not globally
- Can get stuck in loops or produce nonsense
- Has no understanding of meaning - just pattern matching

**The fundamental problem**: N-grams treat words as discrete symbols with no relationship to each other. "cat" and "kitten" are as different as "cat" and "quantum".

---

## Part 2: Word Embeddings (Word2Vec Era, 2013)

**The breakthrough**: Represent words as dense vectors where similar words are close together.

**Key insight**: "You shall know a word by the company it keeps" - words that appear in similar contexts have similar meanings.

**The famous result**: king - man + woman = queen (approximately)

**Limitation**: Each word has ONE embedding, regardless of context. "bank" (river) = "bank" (financial).

In [ ]:
class SimpleWord2Vec:
    """
    Simplified Word2Vec (Skip-gram) implementation.
    
    The idea: Given a center word, predict the surrounding context words.
    Words that appear in similar contexts get similar embeddings.
    """
    
    def __init__(self, embedding_dim: int = 50, window_size: int = 2, learning_rate: float = 0.01):
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.lr = learning_rate
        self.word_to_idx = {}
        self.idx_to_word = {}
        self.W_in = None   # Input embeddings (center words)
        self.W_out = None  # Output embeddings (context words)
    
    def _build_vocab(self, texts: list[str]) -> None:
        """Build vocabulary from texts."""
        words = set()
        for text in texts:
            words.update(text.lower().split())
        
        self.word_to_idx = {w: i for i, w in enumerate(sorted(words))}
        self.idx_to_word = {i: w for w, i in self.word_to_idx.items()}
        
        vocab_size = len(self.word_to_idx)
        # Initialize embeddings randomly
        self.W_in = np.random.randn(vocab_size, self.embedding_dim) * 0.01
        self.W_out = np.random.randn(vocab_size, self.embedding_dim) * 0.01
    
    def _get_training_pairs(self, texts: list[str]) -> list[tuple[int, int]]:
        """Generate (center_word, context_word) pairs."""
        pairs = []
        for text in texts:
            tokens = text.lower().split()
            for i, center in enumerate(tokens):
                center_idx = self.word_to_idx[center]
                # Get context words within window
                start = max(0, i - self.window_size)
                end = min(len(tokens), i + self.window_size + 1)
                for j in range(start, end):
                    if i != j:
                        context_idx = self.word_to_idx[tokens[j]]
                        pairs.append((center_idx, context_idx))
        return pairs
    
    def _sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Sigmoid activation."""
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def fit(self, texts: list[str], epochs: int = 100, negative_samples: int = 5) -> list[float]:
        """Train Word2Vec using negative sampling."""
        self._build_vocab(texts)
        pairs = self._get_training_pairs(texts)
        vocab_size = len(self.word_to_idx)
        
        losses = []
        for epoch in range(epochs):
            epoch_loss = 0
            random.shuffle(pairs)
            
            for center_idx, context_idx in pairs:
                # Positive sample: center and context should be similar
                center_emb = self.W_in[center_idx]
                context_emb = self.W_out[context_idx]
                
                score = np.dot(center_emb, context_emb)
                prob = self._sigmoid(score)
                
                # Loss for positive sample
                epoch_loss += -np.log(prob + 1e-10)
                
                # Gradient for positive sample
                grad = (prob - 1)
                self.W_in[center_idx] -= self.lr * grad * context_emb
                self.W_out[context_idx] -= self.lr * grad * center_emb
                
                # Negative samples: center and random words should be dissimilar
                for _ in range(negative_samples):
                    neg_idx = random.randint(0, vocab_size - 1)
                    if neg_idx == context_idx:
                        continue
                    
                    neg_emb = self.W_out[neg_idx]
                    score = np.dot(center_emb, neg_emb)
                    prob = self._sigmoid(score)
                    
                    epoch_loss += -np.log(1 - prob + 1e-10)
                    
                    grad = prob
                    self.W_in[center_idx] -= self.lr * grad * neg_emb
                    self.W_out[neg_idx] -= self.lr * grad * center_emb
            
            losses.append(epoch_loss / len(pairs))
        
        return losses
    
    def get_embedding(self, word: str) -> np.ndarray:
        """Get the embedding for a word."""
        idx = self.word_to_idx.get(word.lower())
        if idx is None:
            raise ValueError(f"Word '{word}' not in vocabulary")
        return self.W_in[idx]
    
    def most_similar(self, word: str, top_k: int = 5) -> list[tuple[str, float]]:
        """Find most similar words by cosine similarity."""
        word_emb = self.get_embedding(word)
        word_emb = word_emb / (np.linalg.norm(word_emb) + 1e-10)
        
        similarities = []
        for w, idx in self.word_to_idx.items():
            if w == word.lower():
                continue
            emb = self.W_in[idx]
            emb = emb / (np.linalg.norm(emb) + 1e-10)
            sim = np.dot(word_emb, emb)
            similarities.append((w, sim))
        
        return sorted(similarities, key=lambda x: -x[1])[:top_k]
    
    def analogy(self, a: str, b: str, c: str, top_k: int = 3) -> list[tuple[str, float]]:
        """Solve analogy: a is to b as c is to ?"""
        # vector(b) - vector(a) + vector(c) should be close to the answer
        vec = self.get_embedding(b) - self.get_embedding(a) + self.get_embedding(c)
        vec = vec / (np.linalg.norm(vec) + 1e-10)
        
        exclude = {a.lower(), b.lower(), c.lower()}
        similarities = []
        for w, idx in self.word_to_idx.items():
            if w in exclude:
                continue
            emb = self.W_in[idx]
            emb = emb / (np.linalg.norm(emb) + 1e-10)
            sim = np.dot(vec, emb)
            similarities.append((w, sim))
        
        return sorted(similarities, key=lambda x: -x[1])[:top_k]

In [ ]:
# Train Word2Vec on a larger corpus about AI/ML
word2vec_corpus = [
    "the king ruled the kingdom with wisdom",
    "the queen ruled the kingdom with grace", 
    "the man worked in the field",
    "the woman worked in the office",
    "neural networks learn patterns from data",
    "deep learning uses neural networks",
    "machine learning algorithms learn from examples",
    "the model predicts the next word",
    "language models generate text",
    "transformers use attention mechanisms",
    "attention helps models focus on relevant parts",
    "the cat sat on the mat",
    "the dog ran in the park",
    "cats and dogs are pets",
    "pets need food and water",
    "learning requires practice and patience",
    "practice makes perfect",
    "models improve with more data",
    "data is the new oil",
    "artificial intelligence transforms industries",
]

# Train the model
w2v = SimpleWord2Vec(embedding_dim=30, window_size=2, learning_rate=0.05)
losses = w2v.fit(word2vec_corpus, epochs=200, negative_samples=5)

print(f"Training complete! Final loss: {losses[-1]:.4f}")
print(f"Vocabulary size: {len(w2v.word_to_idx)}")

### Word2Vec Limitations

Word2Vec was revolutionary, but it has a critical flaw:

**Static embeddings**: "I went to the bank to deposit money" and "I sat by the river bank" use the SAME embedding for "bank".

**No sequence understanding**: Word2Vec treats sentences as bags of words. Word order doesn't matter for the embeddings.

This led to the next innovation: **sequence models** that process words in order.

---

## Part 3: Recurrent Neural Networks (RNN/LSTM Era)

**The idea**: Process sequences one token at a time, maintaining a "hidden state" that remembers what came before.

**Key insight**: Language is sequential. The meaning of a word depends on what came before it.

**How it works**:
```
Input:  "The"  ->  "cat"  ->  "sat"  ->  "on"
         |          |          |          |
         v          v          v          v
State:  h0   ->    h1   ->    h2   ->    h3
```

Each hidden state `h_t` is a function of the current input AND the previous hidden state.

**Limitation**: 
- Sequential processing = slow (can't parallelize)
- "Vanishing gradients" = hard to learn long-range dependencies
- By the time you reach word 100, you've mostly forgotten word 1

In [ ]:
class SimpleRNN:
    """
    Simple RNN for character-level language modeling.
    
    This demonstrates the core RNN concept: maintaining hidden state
    across a sequence to capture dependencies.
    """
    
    def __init__(self, vocab_size: int, hidden_size: int = 64):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        
        # Initialize weights with small random values
        scale = 0.01
        self.Wxh = np.random.randn(hidden_size, vocab_size) * scale    # input -> hidden
        self.Whh = np.random.randn(hidden_size, hidden_size) * scale   # hidden -> hidden
        self.Why = np.random.randn(vocab_size, hidden_size) * scale    # hidden -> output
        self.bh = np.zeros((hidden_size, 1))                           # hidden bias
        self.by = np.zeros((vocab_size, 1))                            # output bias
    
    def forward(self, inputs: list[int], h_prev: np.ndarray) -> tuple:
        """
        Forward pass through the RNN.
        
        Args:
            inputs: List of character indices
            h_prev: Previous hidden state
        
        Returns:
            outputs: Probability distributions for each position
            hidden_states: All hidden states (for backprop)
            final_h: Final hidden state
        """
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = h_prev.copy()
        
        for t, char_idx in enumerate(inputs):
            # One-hot encode input
            xs[t] = np.zeros((self.vocab_size, 1))
            xs[t][char_idx] = 1
            
            # Hidden state: h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
            hs[t] = np.tanh(self.Wxh @ xs[t] + self.Whh @ hs[t-1] + self.bh)
            
            # Output: y_t = W_hy * h_t + b_y
            ys[t] = self.Why @ hs[t] + self.by
            
            # Softmax for probabilities
            exp_ys = np.exp(ys[t] - np.max(ys[t]))
            ps[t] = exp_ys / np.sum(exp_ys)
        
        return ps, hs, hs[len(inputs) - 1]
    
    def sample(self, seed_char: int, h: np.ndarray, length: int = 100) -> list[int]:
        """Generate a sequence by sampling from the model."""
        x = np.zeros((self.vocab_size, 1))
        x[seed_char] = 1
        indices = []
        
        for _ in range(length):
            # Forward one step
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            
            # Sample from output distribution
            exp_y = np.exp(y - np.max(y))
            p = exp_y / np.sum(exp_y)
            idx = np.random.choice(self.vocab_size, p=p.ravel())
            
            # Prepare next input
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1
            indices.append(idx)
        
        return indices

# Train a character-level RNN on some text
training_text = """
The transformer architecture revolutionized natural language processing.
Attention mechanisms allow models to focus on relevant parts of the input.
Large language models learn patterns from massive amounts of text data.
Neural networks can learn complex representations of language.
Deep learning has transformed how we build AI systems.
"""

# Build character vocabulary
chars = sorted(list(set(training_text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size} characters")
print(f"Characters: {''.join(chars[:20])}...")

# Create the RNN
rnn = SimpleRNN(vocab_size=vocab_size, hidden_size=64)

# Convert text to indices
data = [char_to_idx[ch] for ch in training_text]
print(f"Training data length: {len(data)} characters")

# Generate some text from the untrained RNN (will be random)
h = np.zeros((rnn.hidden_size, 1))
seed_idx = char_to_idx['T']

print("=== Text from UNTRAINED RNN ===")
sample_indices = rnn.sample(seed_idx, h, length=100)
sample_text = ''.join([idx_to_char[i] for i in sample_indices])
print(f"T{sample_text}")
print("\n(This is random because the model hasn't learned anything yet)")

### RNN Limitations Visualized

The key problem with RNNs is the **vanishing gradient** problem:

```
Word 1 -> Word 2 -> Word 3 -> ... -> Word 100 -> Word 101
  |         |         |                 |          |
  v         v         v                 v          v
 h1   ->   h2   ->   h3   -> ... ->   h100  ->   h101
  
Gradient flows backward through ALL these steps.
By the time it reaches Word 1, it's nearly zero!
```

This means RNNs struggle to learn that "The cat, which was sitting on the mat in the living room, **was** happy" - the verb "was" needs to agree with "cat" from many words ago.

**LSTM and GRU** partially solved this with "gates" that control information flow, but the fundamental sequential bottleneck remained.

---

## Part 4: The Transformer Revolution (2017)

**The breakthrough**: "Attention Is All You Need" - process ALL tokens in parallel using attention.

**Key insight**: Instead of passing information through a chain of hidden states, let every token directly "look at" every other token.

**Why this changed everything**:
1. **Parallelization**: No sequential dependency = train on GPUs efficiently
2. **Long-range dependencies**: Token 1 can directly attend to token 100
3. **Scalability**: Performance improves predictably with more data and compute

Let's build a mini transformer to see how attention works!

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class MiniTransformer:
    """
    A minimal transformer implementation to demonstrate self-attention.
    
    This is a simplified single-layer, single-head transformer for educational purposes.
    Real transformers have multiple layers, multiple attention heads, and more.
    """
    
    def __init__(self, vocab_size: int, d_model: int = 32, context_length: int = 16):
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.context_length = context_length
        
        # Token embeddings
        self.token_emb = np.random.randn(vocab_size, d_model) * 0.02
        
        # Positional embeddings (learned, not sinusoidal for simplicity)
        self.pos_emb = np.random.randn(context_length, d_model) * 0.02
        
        # Attention weights: Query, Key, Value projections
        self.W_q = np.random.randn(d_model, d_model) * 0.02
        self.W_k = np.random.randn(d_model, d_model) * 0.02
        self.W_v = np.random.randn(d_model, d_model) * 0.02
        
        # Output projection
        self.W_out = np.random.randn(d_model, vocab_size) * 0.02
    
    def attention(self, x: np.ndarray, return_weights: bool = False):
        """
        Self-attention mechanism.
        
        Args:
            x: Input embeddings (seq_len, d_model)
            return_weights: If True, also return attention weights
        
        Returns:
            Output after attention (seq_len, d_model)
        """
        seq_len = x.shape[0]
        
        # Compute Q, K, V
        Q = x @ self.W_q  # (seq_len, d_model)
        K = x @ self.W_k
        V = x @ self.W_v
        
        # Attention scores: Q @ K^T / sqrt(d_model)
        scores = (Q @ K.T) / np.sqrt(self.d_model)  # (seq_len, seq_len)
        
        # Causal mask: prevent attending to future tokens
        # This is what makes it a "decoder" / autoregressive model
        mask = np.triu(np.ones((seq_len, seq_len)) * -1e9, k=1)
        scores = scores + mask
        
        # Softmax to get attention weights
        attn_weights = softmax(scores, axis=-1)  # (seq_len, seq_len)
        
        # Weighted sum of values
        output = attn_weights @ V  # (seq_len, d_model)
        
        if return_weights:
            return output, attn_weights
        return output
    
    def forward(self, token_indices: list[int], return_attention: bool = False):
        """
        Forward pass through the mini transformer.
        
        Args:
            token_indices: List of token indices
            return_attention: If True, also return attention weights
        
        Returns:
            logits: (seq_len, vocab_size) - unnormalized probabilities
        """
        seq_len = len(token_indices)
        
        # Get token embeddings
        x = self.token_emb[token_indices]  # (seq_len, d_model)
        
        # Add positional embeddings
        x = x + self.pos_emb[:seq_len]
        
        # Self-attention
        if return_attention:
            x, attn_weights = self.attention(x, return_weights=True)
        else:
            x = self.attention(x)
            attn_weights = None
        
        # Project to vocabulary
        logits = x @ self.W_out  # (seq_len, vocab_size)
        
        if return_attention:
            return logits, attn_weights
        return logits
    
    def generate(self, start_tokens: list[int], max_new_tokens: int = 50) -> list[int]:
        """Generate tokens autoregressively."""
        tokens = start_tokens.copy()
        
        for _ in range(max_new_tokens):
            # Only use last context_length tokens
            context = tokens[-self.context_length:]
            
            # Get logits for next token
            logits = self.forward(context)
            
            # Sample from the last position
            probs = softmax(logits[-1])
            next_token = np.random.choice(self.vocab_size, p=probs)
            
            tokens.append(next_token)
        
        return tokens

# Create a mini transformer and visualize attention
transformer = MiniTransformer(vocab_size=vocab_size, d_model=32, context_length=32)

# Use a sample sentence
sample_sentence = "The transformer"
sample_indices = [char_to_idx[ch] for ch in sample_sentence]

# Get attention weights
logits, attn_weights = transformer.forward(sample_indices, return_attention=True)

print(f"Input: '{sample_sentence}'")
print(f"Input length: {len(sample_indices)} characters")
print(f"Attention matrix shape: {attn_weights.shape}")
print("\nAttention weights (who attends to whom):")
print("Rows = query position, Columns = key position")
print("(Causal mask: can only attend to previous positions)\n")

# Print attention matrix with character labels
chars_in_sample = list(sample_sentence)
print("     ", end="")
for ch in chars_in_sample:
    print(f"{ch:>5}", end="")
print()

for i, ch in enumerate(chars_in_sample):
    print(f"{ch:>3}: ", end="")
    for j in range(len(chars_in_sample)):
        print(f"{attn_weights[i, j]:>5.2f}", end="")
    print()

# Visualize attention with matplotlib (if available)
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(attn_weights, cmap='Blues')
    
    ax.set_xticks(range(len(chars_in_sample)))
    ax.set_yticks(range(len(chars_in_sample)))
    ax.set_xticklabels(chars_in_sample)
    ax.set_yticklabels(chars_in_sample)
    
    ax.set_xlabel('Key position (attending to)')
    ax.set_ylabel('Query position (from)')
    ax.set_title('Self-Attention Weights (Causal Mask)')
    
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not installed - skipping visualization")

### Understanding the Attention Pattern

Notice the **triangular pattern** in the attention matrix:
- Each position can only attend to itself and previous positions (causal mask)
- This is what makes it "autoregressive" - predicting the next token based only on past tokens
- Position 0 ("T") can only attend to itself
- Position 5 ("a") can attend to positions 0-5

**This is exactly how GPT, Claude, and other decoder-only LLMs work!**

The key difference from RNNs:
- **RNN**: Information flows sequentially through hidden states
- **Transformer**: Every position can directly attend to every previous position in ONE step

---

## Part 5: Why Transformers Won

Let's compare the three approaches we've built:

| Aspect | N-gram | RNN | Transformer |
|--------|--------|-----|-------------|
| **Context** | N-1 words | All previous (in theory) | All previous (in practice) |
| **Parallelization** | N/A | No (sequential) | Yes (all at once) |
| **Long-range deps** | None | Weak (vanishing gradient) | Strong (direct attention) |
| **Training speed** | Fast | Slow | Fast (on GPUs) |
| **Scalability** | Poor | Moderate | Excellent |

### The Scaling Laws Discovery

In 2020, OpenAI discovered that transformer performance follows predictable **scaling laws**:

```
Performance improves smoothly as you increase:
1. Model size (parameters)
2. Dataset size (tokens)
3. Compute (FLOPs)
```

This meant: if you want a better model, just make it bigger and train it on more data. No architectural changes needed.

This insight led to:
- GPT-3 (175B parameters)
- GPT-4 (rumored ~1T parameters)
- Claude, Gemini, Llama, and others

---

## Part 6: Building a Mini GPT (Putting It All Together)

Now let's use PyTorch to build a more complete mini-GPT that we can actually train. This demonstrates the full pipeline:

1. Tokenization
2. Embedding + Positional Encoding
3. Multi-head Self-Attention
4. Feed-forward Network
5. Next-token Prediction

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Check if PyTorch is available and set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
class MiniGPT(nn.Module):
    """
    A minimal GPT-style transformer for educational purposes.
    
    Architecture:
    - Token + Positional Embeddings
    - N Transformer Blocks (each with Self-Attention + FFN)
    - Output projection to vocabulary
    """
    
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        context_length: int = 64,
        dropout: float = 0.1
    ):
        super().__init__()
        self.context_length = context_length
        
        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm and output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Token + positional embeddings
        tok_emb = self.token_emb(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Final layer norm and projection
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        
        # Compute loss if targets provided
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        else:
            loss = None
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generate new tokens autoregressively."""
        for _ in range(max_new_tokens):
            # Crop to context length
            idx_cond = idx[:, -self.context_length:]
            
            # Get predictions
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            # Sample from distribution
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            idx = torch.cat([idx, idx_next], dim=1)
        
        return idx


class TransformerBlock(nn.Module):
    """A single transformer block: attention + feedforward."""
    
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Pre-norm architecture (like GPT-2)
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with causal mask."""
    
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Compute Q, K, V
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        # Reshape for multi-head attention
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        
        # Softmax and apply to values
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ v
        
        # Reshape back
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class FeedForward(nn.Module):
    """Feed-forward network with GELU activation."""
    
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

# Prepare training data for Mini GPT
# We'll train on a small corpus about AI/ML

gpt_training_text = """
Large language models are trained on massive amounts of text data.
The transformer architecture uses self-attention to process sequences.
Attention allows each token to look at all previous tokens directly.
Neural networks learn patterns from data through gradient descent.
Deep learning has revolutionized natural language processing.
GPT models predict the next token given the previous context.
Training involves minimizing the cross-entropy loss function.
The model learns to generate coherent and contextual text.
Transfer learning enables adapting pretrained models to new tasks.
Fine-tuning adjusts model weights for specific applications.
"""

# Character-level tokenization for simplicity
gpt_chars = sorted(list(set(gpt_training_text)))
gpt_char_to_idx = {ch: i for i, ch in enumerate(gpt_chars)}
gpt_idx_to_char = {i: ch for ch, i in gpt_char_to_idx.items()}
gpt_vocab_size = len(gpt_chars)

# Encode the text
gpt_data = torch.tensor([gpt_char_to_idx[ch] for ch in gpt_training_text], dtype=torch.long)

print(f"Vocabulary size: {gpt_vocab_size}")
print(f"Training data: {len(gpt_data)} characters")
print(f"Sample: '{gpt_training_text[:50]}...'")

# Create the Mini GPT model
model = MiniGPT(
    vocab_size=gpt_vocab_size,
    d_model=64,
    n_heads=4,
    n_layers=2,
    context_length=32,
    dropout=0.1
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Create data loader
def get_batch(data, batch_size=32, context_length=32):
    """Get a random batch of training data."""
    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x.to(device), y.to(device)

# Train the Mini GPT!
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Training Mini GPT...")
print("=" * 50)

losses = []
for step in range(500):
    # Get batch
    xb, yb = get_batch(gpt_data, batch_size=16, context_length=32)
    
    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.4f}")

print("=" * 50)
print(f"Final loss: {losses[-1]:.4f}")

# Generate text from the trained model!
print("=== Text Generation from Trained Mini GPT ===\n")

prompts = ["The ", "Neural ", "Large "]

for prompt in prompts:
    # Encode prompt
    prompt_idx = torch.tensor([[gpt_char_to_idx[ch] for ch in prompt]], dtype=torch.long).to(device)
    
    # Generate
    model.eval()
    with torch.no_grad():
        generated = model.generate(prompt_idx, max_new_tokens=100, temperature=0.8)
    
    # Decode
    generated_text = ''.join([gpt_idx_to_char[i.item()] for i in generated[0]])
    print(f"Prompt: '{prompt}'")
    print(f"Generated: {generated_text[:150]}...")
    print()

---

## Summary: The Evolution of Language Models

We've built working examples of each major paradigm:

| Era | Model | Key Innovation | Limitation |
|-----|-------|----------------|------------|
| **Statistical** | N-gram | Count-based probabilities | No semantics, limited context |
| **Embeddings** | Word2Vec | Dense word vectors | Static (no context sensitivity) |
| **Sequential** | RNN/LSTM | Hidden state memory | Slow, vanishing gradients |
| **Attention** | Transformer | Parallel attention | Compute scales quadratically |

### Why Transformers Dominate

1. **Parallelization**: Train efficiently on GPUs
2. **Long-range dependencies**: Direct attention to any previous token
3. **Scalability**: Predictable improvement with scale
4. **Flexibility**: Same architecture works for text, code, images, audio

### What Makes Modern LLMs Special

The models we built are tiny (~50K parameters). GPT-4 has ~1 trillion parameters.

But the core mechanism is the same:
- Token embeddings
- Self-attention (who should I pay attention to?)
- Feed-forward networks (what should I compute?)
- Next-token prediction

**Scale + Data + Architecture = Modern LLMs**

---

## Exercises

1. **N-gram experiments**: Modify the N-gram model to use n=4 (4-gram). How does generation quality change? What about perplexity?

2. **Word2Vec exploration**: Add more sentences to the Word2Vec corpus about a specific domain (e.g., cooking, sports). Do the embeddings capture domain-specific relationships?

3. **Attention analysis**: Modify the MiniTransformer to return attention weights during generation. Which characters does the model attend to most when generating the next character?

4. **Scale experiment**: Increase the Mini GPT's `d_model` to 128 and `n_layers` to 4. How does training loss change? How does generation quality improve?

5. **Temperature exploration**: Generate text with temperature values of 0.5, 1.0, and 1.5. How does temperature affect the creativity vs. coherence tradeoff?

---

## References

- **Attention Is All You Need** (Vaswani et al., 2017) - The original transformer paper
- **Word2Vec** (Mikolov et al., 2013) - Efficient estimation of word representations
- **GPT-2** (Radford et al., 2019) - Language models are unsupervised multitask learners
- **Scaling Laws** (Kaplan et al., 2020) - Scaling laws for neural language models
- **The Illustrated Transformer** (Jay Alammar) - Visual explanation of transformers

In [ ]:
# Explore the learned embeddings
print("=== Most Similar Words ===\n")

test_words = ["learning", "model", "data", "neural"]
for word in test_words:
    try:
        similar = w2v.most_similar(word, top_k=3)
        print(f"'{word}' is similar to:")
        for w, score in similar:
            print(f"  - {w}: {score:.3f}")
        print()
    except ValueError as e:
        print(f"  {e}\n")

# Try word analogies (may not work perfectly with small corpus)
print("=== Word Analogies ===")
print("king - man + woman = ?")
try:
    results = w2v.analogy("man", "king", "woman", top_k=3)
    for w, score in results:
        print(f"  - {w}: {score:.3f}")
except ValueError as e:
    print(f"  {e}")